<a href="https://colab.research.google.com/github/korzhimanov/dsp-seminars/blob/main/seminars/6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Практическое занятие №6: Синтез цифровых фильтров (КИХ и БИХ)

## Часть 1. Синтез КИХ-фильтров методом окон

### Задание 1.1. Проектирование ФНЧ с разными окнами
Спроектируйте КИХ-ФНЧ с частотой среза 200 Гц (частота дискретизации 1000 Гц), длина фильтра 51 отсчёт. Используйте окна:
- прямоугольное,
- Ханна,
- Хемминга,
- Блэкмана.

Постройте на одном графике АЧХ (в дБ) всех четырёх фильтров. Сравните:
- крутизну среза,
- уровень пульсаций в полосе пропускания и заграждения,
- ширину переходной полосы.


In [2]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)

fs = 1000
fc = 200
numtaps = 51


def plot_db_response(b, a=[1], fs=1000, label=None):
    w, h = signal.freqz(b, a, worN=4096, fs=fs)
    mag_db = 20 * np.log10(np.abs(h) + 1e-12)
    plt.plot(w, mag_db, label=label)
    return w, mag_db


windows = ['boxcar', 'hann', 'hamming', 'blackman']
fir_windows = {}

plt.figure(figsize=(12, 5))
for win in windows:
    b = signal.firwin(numtaps, fc, window=win, fs=fs)
    fir_windows[win] = b
    plot_db_response(b, fs=fs, label=win)

plt.title('АЧХ ФНЧ при разных окнах')
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда (дБ)')
plt.ylim(-120, 5)
plt.grid(True)
plt.legend()
plt.show()



**Вопросы:** Какое окно обеспечивает наилучшее подавление в полосе заграждения? Какое – самую крутую переходную полосу? Как это связано с формой окна?

1) Лучшее подавление в полосе заграждения даёт окно Блэкмана
2) Самая узкая переходная полоса у прямоугольного окна, но у него самые сильные боковые лепестки
3) Чем сильнее окно сглажено по краям, тем ниже боковые лепестки, но шире переходная полоса


### Задание 1.2. Влияние длины фильтра
Для окна Хемминга спроектируйте ФНЧ с fc=200 Гц, fs=1000 Гц, с длинами 21, 51, 101. Постройте АЧХ на одном графике.


In [95]:
lengths = [21, 51, 101]
hamming_filters = {}

plt.figure(figsize=(12, 5))
for L in lengths:
    b = signal.firwin(L, fc, window='hamming', fs=fs)
    hamming_filters[L] = b
    plot_db_response(b, fs=fs, label=f'L = {L}')

plt.title('Влияние длины КИХ-фильтра')
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда (дБ)')
plt.ylim(-120, 5)
plt.grid(True)
plt.legend()
plt.show()



**Вопрос:** Как увеличение длины влияет на крутизну среза и на уровень боковых лепестков?

При увеличении длины фильтра срез становится круче, а переходная область уже. Уровень боковых лепестков для того же окна меняется не так сильно, но сами лепестки становятся уже



### Задание 1.3. Синтез ФВЧ и полосового фильтра методом окон
Используя тот же оконный метод, спроектируйте:
- ФВЧ с fc=200 Гц (длина 51, окно Хемминга);
- полосовой фильтр с полосой пропускания 200–300 Гц (длина 51, окно Хемминга).

Постройте АЧХ обоих фильтров.


In [94]:
b_high = signal.firwin(51, fc, window='hamming', fs=fs, pass_zero=False)
b_band = signal.firwin(51, [200, 300], window='hamming', fs=fs, pass_zero=False)

plt.figure(figsize=(12, 5))
plot_db_response(b_high, fs=fs, label='ФВЧ fc = 200 Гц')
plot_db_response(b_band, fs=fs, label='Полосовой 200-300 Гц')
plt.title('АЧХ ФВЧ и полосового фильтра')
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда (дБ)')
plt.ylim(-120, 5)
plt.grid(True)
plt.legend()
plt.show()



Подайте на спроектированные фильтры сигнал: сумма синусоид 100, 250 и 350 Гц (амплитуды 1, 0.8, 0.6). Постройте спектры до и после фильтрации (в логарифмическом масштабе).

In [93]:
t = np.arange(0, 1, 1 / fs)
x = (
    np.sin(2 * np.pi * 100 * t)
    + 0.8 * np.sin(2 * np.pi * 250 * t)
    + 0.6 * np.sin(2 * np.pi * 350 * t)
)

y_high = signal.lfilter(b_high, [1], x)
y_band = signal.lfilter(b_band, [1], x)


def plot_spectrum(x, label):
    freqs = np.fft.rfftfreq(len(x), 1 / fs)
    X = np.fft.rfft(x)
    amp = 2 * np.abs(X) / len(x)
    plt.semilogy(freqs, amp + 1e-6, label=label)


plt.figure(figsize=(12, 5))
plot_spectrum(x, 'Исходный')
plot_spectrum(y_high, 'После ФВЧ')
plot_spectrum(y_band, 'После полосового')
plt.xlim(0, 500)
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.legend()
plt.show()



**Вопрос:** Объясните полученные результаты.

1) ФВЧ подавляет 100 Гц и пропускает 250 и 350 Гц
2) Полосовой фильтр лучше всего пропускает 250 Гц, а 100 и 350 Гц подавляет
3) Это соответствует их полосам пропускания


## Часть 2. Равноволновой синтез КИХ-фильтров (Паркса–МакКлеллана)

### Задание 2.1. Проектирование оптимального ФНЧ
Используя `signal.remez`, спроектируйте ФНЧ с параметрами:
- частота дискретизации 1000 Гц,
- полоса пропускания 0–150 Гц,
- полоса заграждения 250–500 Гц,
- длина фильтра 21.

Постройте АЧХ (в линейном и логарифмическом масштабах) и сравните её с АЧХ КИХ-фильтра, полученного методом окон (окно Хемминга, та же длина, fc=200 Гц).

In [92]:
numtaps = 21
b_remez = signal.remez(numtaps, [0, 150, 250, 500], [1, 0], fs=fs)
b_hamming_21 = signal.firwin(numtaps, 200, window='hamming', fs=fs)

plt.figure(figsize=(12, 5))
plot_db_response(b_remez, fs=fs, label='remez')
plot_db_response(b_hamming_21, fs=fs, label='hamming')
plt.title('Сравнение remez и оконного фильтра')
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда (дБ)')
plt.ylim(-100, 5)
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
for b, label in [(b_remez, 'remez'), (b_hamming_21, 'hamming')]:
    w, h = signal.freqz(b, [1], worN=4096, fs=fs)
    plt.plot(w, np.abs(h), label=label)
plt.title('АЧХ в линейном масштабе')
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда')
plt.grid(True)
plt.legend()
plt.show()




**Вопросы:** Какой фильтр имеет более крутой срез? Каковы пульсации в полосе пропускания?

1) Более крутой срез имеет фильтр remez
2) У remez пульсации в полосе пропускания заметнее и распределены почти равномерно
3) У оконного фильтра Хемминга АЧХ более гладкая, но переходная область шире


### Задание 2.2. Зависимость от ширины переходной полосы

Поменяйте верхнюю частоту полосы пропускания и нижнюю частоту полосы заграждения так, чтобы их среднее оставалось равным 200 Гц. Постройте АЧХ получившегося фильтра. Подберите ширину переходной области (разности нижней частоты полосы заграждения и верхней частоты полосы пропускания) так, чтобы уровень подавления в полосе заграждения у равноволнового фильтра совпал с оконным фильтром Хемминга.


In [ ]:
widths = [40, 60, 80, 100, 120, 140, 160]
stop_levels = []

w_ham, h_ham = signal.freqz(b_hamming_21, [1], worN=4096, fs=fs)
ham_stop = np.max(20 * np.log10(np.abs(h_ham[w_ham >= 250]) + 1e-12))

plt.figure(figsize=(12, 5))
for width in widths:
    fp = 200 - width / 2
    fs_stop = 200 + width / 2
    b = signal.remez(21, [0, fp, fs_stop, 500], [1, 0], fs=fs)
    w, h = signal.freqz(b, [1], worN=4096, fs=fs)
    mag_db = 20 * np.log10(np.abs(h) + 1e-12)
    stop_level = np.max(mag_db[w >= fs_stop])
    stop_levels.append(stop_level)
    plt.plot(w, mag_db, label=f'width = {width}')

plt.axhline(ham_stop, color='k', linestyle='--', label='hamming stop')
plt.title('remez при разной ширине переходной области')
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда (дБ)')
plt.ylim(-100, 5)
plt.grid(True)
plt.legend()
plt.show()

for width, level in zip(widths, stop_levels):
    print(f'width = {width}, stop level = {level:.2f} dB')
print(f'hamming stop level = {ham_stop:.2f} dB')



**Вопрос:** При какой ширине переходной области уровень подавления в полосе заграждения у равноволнового фильтра совпал с оконным фильтром Хемминга?

Уровень подавления получается ближе всего к оконному фильтру Хемминга при ширине переходной области около 40 Гц


### Задание 2.3. Управление весами
Для фильтра из задачи 2.1 измените весовые коэффициенты: задайте `weight=[1, 10]` (увеличить вес для полосы заграждения). Постройте новую АЧХ и сравните с предыдущей.


In [91]:
b_remez_1 = signal.remez(21, [0, 150, 250, 500], [1, 0], weight=[1, 1], fs=fs)
b_remez_10 = signal.remez(21, [0, 150, 250, 500], [1, 0], weight=[1, 10], fs=fs)

plt.figure(figsize=(12, 5))
plot_db_response(b_remez_1, fs=fs, label='weight = [1, 1]')
plot_db_response(b_remez_10, fs=fs, label='weight = [1, 10]')
plt.title('Влияние весов в remez')
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда (дБ)')
plt.ylim(-100, 5)
plt.grid(True)
plt.legend()
plt.show()



**Вопрос:** Как изменилось подавление в полосе заграждения и пульсации в полосе пропускания?

При увеличении веса полосы заграждения подавление в ней стало сильнее, но пульсации в полосе пропускания стали больше


### Задание 2.4. Сравнение КИХ-фильтров на реальном сигнале
Сгенерируйте сигнал: смесь синусоид 50 Гц, 120 Гц, 220 Гц (амплитуды 1, 0.7, 0.3) + белый шум (дисперсия 0.1), fs=1000 Гц. Пропустите этот сигнал через:
- КИХ-ФНЧ спроектированный методом окон (окно Хемминга, длина 51, fc=150 Гц);
- КИХ-ФНЧ спроектированный методом `remez` (длина 51, полоса пропускания 0–100 Гц, полоса заграждения 200–500 Гц).

Постройте спектры исходного и отфильтрованных сигналов в линейном и логарифмическом масштабе.


In [90]:
np.random.seed(42)
fs = 1000
t = np.arange(0, 1, 1 / fs)
x = (
    np.sin(2 * np.pi * 50 * t)
    + 0.7 * np.sin(2 * np.pi * 120 * t)
    + 0.3 * np.sin(2 * np.pi * 220 * t)
    + np.random.normal(0, np.sqrt(0.1), len(t))
)

b_win = signal.firwin(51, 150, window='hamming', fs=fs)
b_rem = signal.remez(51, [0, 100, 200, 500], [1, 0], fs=fs)

y_win = signal.lfilter(b_win, [1], x)
y_rem = signal.lfilter(b_rem, [1], x)


def amp_at(x, freq):
    freqs = np.fft.rfftfreq(len(x), 1 / fs)
    X = np.fft.rfft(x)
    idx = np.argmin(np.abs(freqs - freq))
    return 2 * np.abs(X[idx]) / len(x)


plt.figure(figsize=(12, 5))
plot_spectrum(x, 'Исходный')
plot_spectrum(y_win, 'Оконный')
plot_spectrum(y_rem, 'remez')
plt.xlim(0, 500)
plt.title('Спектры сигналов')
plt.grid(True)
plt.legend()
plt.show()

print(f'Исходный A120 = {amp_at(x, 120):.3f}')
print(f'Оконный A120 = {amp_at(y_win, 120):.3f}')
print(f'remez A120 = {amp_at(y_rem, 120):.3f}')
print(f'Оконный A220 = {amp_at(y_win, 220):.3f}')
print(f'remez A220 = {amp_at(y_rem, 220):.3f}')



**Вопросы:** Какой фильтр лучше подавил 220 Гц и шум? Какой лучше сохранил 120 Гц? На сколько отличаются амплитуды 120 Гц сигнала от исходных для каждого из фильтров?


1) Оба фильтра хорошо подавили 220 Гц и высокочастотный шум
2) 120 Гц лучше сохранил оконный фильтр: амплитуда получилась около 0.699 вместо 0.717 у исходного сигнала
3) У remez амплитуда 120 Гц получилась около 0.681, то есть потери чуть больше


## Часть 3. Синтез БИХ-фильтров

### Задание 3.1. Баттерворт, Чебышев, эллиптический – сравнение АЧХ
Спроектируйте ФНЧ с частотой среза 200 Гц (fs=1000 Гц) следующих типов (порядок 4):
- Баттерворт (`butter`);
- Чебышев I с пульсациями 1 дБ (`cheby1`);
- Чебышев II с затуханием 40 дБ в полосе заграждения (`cheby2`);
- Эллиптический с пульсациями 1 дБ и затуханием 40 дБ (`ellip`).

Постройте АЧХ всех фильтров на одном графике (в дБ). Сравните:
- крутизну среза,
- пульсации в полосе пропускания и заграждения.

Постройте и сравните фазовые характеристики (ФЧХ) и групповые задержки.


In [89]:
fs = 1000
filters_iir = {
    'butter': signal.butter(4, 200, fs=fs, btype='low'),
    'cheby1': signal.cheby1(4, 1, 200, fs=fs, btype='low'),
    'cheby2': signal.cheby2(4, 40, 200, fs=fs, btype='low'),
    'ellip': signal.ellip(4, 1, 40, 200, fs=fs, btype='low'),
}

plt.figure(figsize=(12, 5))
for name, (b, a) in filters_iir.items():
    plot_db_response(b, a, fs=fs, label=name)
plt.title('АЧХ БИХ-фильтров')
plt.xlabel('Частота (Гц)')
plt.ylabel('Амплитуда (дБ)')
plt.ylim(-100, 5)
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
for name, (b, a) in filters_iir.items():
    w, h = signal.freqz(b, a, worN=4096, fs=fs)
    plt.plot(w, np.unwrap(np.angle(h)), label=name)
plt.title('ФЧХ')
plt.grid(True)
plt.legend()

plt.subplot(2, 1, 2)
for name, (b, a) in filters_iir.items():
    w_gd, gd = signal.group_delay((b, a), fs=fs)
    plt.plot(w_gd, gd, label=name)
plt.title('Групповая задержка')
plt.xlabel('Частота (Гц)')
plt.ylim(-5, 30)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()



**Вопросы:** Какой фильтр даёт самый крутой спад? Какой имеет наименьшие фазовые искажения в полосе пропускания?


1) Самый крутой спад около частоты среза даёт эллиптический фильтр
2) Наименьшие фазовые искажения в полосе пропускания у Баттерворта, потому что его групповая задержка меняется плавнее


### Задание 3.2. Влияние порядка на характеристики БИХ-фильтра
Для фильтра Баттерворта с fc=200 Гц, fs=1000 Гц возьмите порядки 2, 4, 6. Постройте АЧХ и групповую задержку.


In [88]:
orders = [2, 4, 6]

plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
for order in orders:
    b, a = signal.butter(order, 200, fs=fs)
    w, h = signal.freqz(b, a, worN=4096, fs=fs)
    plt.plot(w, 20 * np.log10(np.abs(h) + 1e-12), label=f'order = {order}')
plt.title('АЧХ Баттерворта')
plt.ylabel('Амплитуда (дБ)')
plt.ylim(-100, 5)
plt.grid(True)
plt.legend()

plt.subplot(2, 1, 2)
for order in orders:
    b, a = signal.butter(order, 200, fs=fs)
    w_gd, gd = signal.group_delay((b, a), fs=fs)
    plt.plot(w_gd, gd, label=f'order = {order}')
plt.title('Групповая задержка')
plt.xlabel('Частота (Гц)')
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()



**Вопросы:** Как увеличение порядка влияет на крутизну среза и на групповую задержку в полосе пропускания?

1) При увеличении порядка срез становится круче и подавление после частоты среза сильнее
2) Групповая задержка в полосе пропускания растёт и становится менее ровной, поэтому фазовые искажения увеличиваются


### Задание 3.3. Преобразование типа БИХ-фильтра (ФНЧ → ФВЧ, полосовой)
Спроектируйте Баттерворта 4-го порядка:
- ФНЧ с fc=200 Гц;
- ФВЧ с fc=200 Гц (используйте `btype='high'`);
- полосовой с полосой 200–300 Гц.

Подайте на них сигнал: сумма синусоид 100, 250, 400 Гц. Постройте спектры после фильтрации.


In [87]:
fs = 1000
t = np.arange(0, 1, 1 / fs)
x = (
    np.sin(2 * np.pi * 100 * t)
    + np.sin(2 * np.pi * 250 * t)
    + np.sin(2 * np.pi * 400 * t)
)

b_low, a_low = signal.butter(4, 200, fs=fs, btype='low')
b_high_iir, a_high_iir = signal.butter(4, 200, fs=fs, btype='high')
b_band_iir, a_band_iir = signal.butter(4, [200, 300], fs=fs, btype='band')

y_low = signal.lfilter(b_low, a_low, x)
y_high_iir = signal.lfilter(b_high_iir, a_high_iir, x)
y_band_iir = signal.lfilter(b_band_iir, a_band_iir, x)

plt.figure(figsize=(12, 5))
plot_spectrum(y_low, 'ФНЧ')
plot_spectrum(y_high_iir, 'ФВЧ')
plot_spectrum(y_band_iir, 'Полосовой')
plt.xlim(0, 500)
plt.title('Спектры после БИХ-фильтров')
plt.grid(True)
plt.legend()
plt.show()



**Вопрос:** Какие составляющие подавлены, а какие пропущены в каждом случае?

1) ФНЧ пропускает 100 Гц и подавляет 250 и 400 Гц
2) ФВЧ подавляет 100 Гц и пропускает 250 и 400 Гц
3) Полосовой фильтр пропускает 250 Гц и подавляет 100 и 400 Гц


### Задание 3.4. Применение БИХ-фильтра к зашумлённому сигналу
Сгенерируйте сигнал: синусоида 50 Гц + белый шум с дисперсией 0.2, fs=1000 Гц, длительность 1 с. Спроектируйте эллиптический ФНЧ с fc=100 Гц (порядок 6, пульсации 1 дБ, затухание 40 дБ). Примените фильтр с помощью `lfilter` и `filtfilt`. Постройте на одном графике исходный и отфильтрованные сигналы (временные области, первые 0.2 с), а также их спектры. Сравните задержку и подавление шума.


In [ ]:
np.random.seed(42)
fs = 1000
t = np.arange(0, 1, 1 / fs)
x_clean = np.sin(2 * np.pi * 50 * t)
x = x_clean + np.random.normal(0, np.sqrt(0.2), len(t))

b_ellip, a_ellip = signal.ellip(6, 1, 40, 100, fs=fs, btype='low')
y_lfilter = signal.lfilter(b_ellip, a_ellip, x)
y_filtfilt = signal.filtfilt(b_ellip, a_ellip, x)

mask = t <= 0.2
plt.figure(figsize=(12, 5))
plt.plot(t[mask], x[mask], 'gray', alpha=0.4, label='Зашумлённый')
plt.plot(t[mask], x_clean[mask], 'k--', label='Чистый')
plt.plot(t[mask], y_lfilter[mask], label='lfilter')
plt.plot(t[mask], y_filtfilt[mask], label='filtfilt')
plt.title('Фильтрация во временной области')
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(12, 5))
plot_spectrum(x, 'Зашумлённый')
plot_spectrum(y_lfilter, 'lfilter')
plot_spectrum(y_filtfilt, 'filtfilt')
plt.xlim(0, 300)
plt.title('Спектры')
plt.grid(True)
plt.legend()
plt.show()



**Вопрос:** Почему `filtfilt` даёт нулевой фазовый сдвиг?

filtfilt фильтрует сигнал два раза: сначала вперёд, потом назад. Из-за этого фазовые сдвиги взаимно компенсируются, поэтому итоговый сигнал не смещается по времени


## Часть 4. Сравнение КИХ и БИХ фильтров одинакового порядка

### Задание 4.1. Сравнение характеристик
Спроектируйте:
- КИХ-ФНЧ методом окон (окно Хемминга, длина 51, fc=200 Гц);
- БИХ-ФНЧ Баттерворта 4-го порядка (fc=200 Гц).

Постройте для них на одном графике:
- отдельно АЧХ (в дБ),
- отдельно групповую задержку.


In [86]:
fs = 1000
b_fir = signal.firwin(51, 200, window='hamming', fs=fs)
b_iir, a_iir = signal.butter(4, 200, fs=fs)

plt.figure(figsize=(12, 8))
plt.subplot(2, 1, 1)
plot_db_response(b_fir, fs=fs, label='КИХ Хемминг')
plot_db_response(b_iir, a_iir, fs=fs, label='БИХ Баттерворт 4')
plt.title('АЧХ')
plt.ylabel('Амплитуда (дБ)')
plt.ylim(-100, 5)
plt.grid(True)
plt.legend()

plt.subplot(2, 1, 2)
w_fir, gd_fir = signal.group_delay((b_fir, [1]), fs=fs)
w_iir, gd_iir = signal.group_delay((b_iir, a_iir), fs=fs)
plt.plot(w_fir, gd_fir, label='КИХ Хемминг')
plt.plot(w_iir, gd_iir, label='БИХ Баттерворт 4')
plt.title('Групповая задержка')
plt.xlabel('Частота (Гц)')
plt.ylim(0, 40)
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

for order in [4, 6, 8, 10, 12]:
    b, a = signal.butter(order, 200, fs=fs)
    w, h = signal.freqz(b, a, worN=4096, fs=fs)
    mag_db = 20 * np.log10(np.abs(h) + 1e-12)
    idx = np.where((w >= 200) & (mag_db <= -40))[0]
    f40 = w[idx[0]] if len(idx) else np.nan
    print(f'order = {order}, f40 = {f40:.1f} Гц')



**Вопросы:** Какой фильтр имеет более крутой срез? Какой имеет постоянную групповую задержку? Какой вносит меньшие фазовые искажения? Какой требует меньше вычислений? Попробуйте увеличить порядок фильтра Баттерворта. При каком порядке крутизна среза его АЧХ становится сравнимой с КИХ-фильтром? Растут или уменьшаются при этом фазовые искажения?


1) При 4-м порядке более крутой срез у КИХ-фильтра
2) Постоянная групповая задержка у КИХ-фильтра, поэтому фазовые искажения меньше у него
3) Меньше вычислений требует БИХ-фильтр, потому что у него меньше коэффициентов
4) У Баттерворта крутизна становится ближе к КИХ примерно при порядке 12, но фазовые искажения при этом растут


### Задание 4.2. Применение к реальному сигналу
Сгенерируйте сигнал: короткий прямоугольный импульс длительностью 10 мс, а также высокочастотная синусоидальная помеха 400 Гц. Частоту дискретизации возьмите fs=10000 Гц.

Примените оба фильтра (КИХ и БИХ из предыдущего пункта). Постройте исходный и отфильтрованные сигналы.


In [85]:
fs = 10000
t = np.arange(0, 0.1, 1 / fs)

pulse = np.zeros_like(t)
pulse[(t >= 0.02) & (t < 0.03)] = 1
interference = 0.4 * np.sin(2 * np.pi * 400 * t)
x = pulse + interference

b_fir = signal.firwin(51, 200, window='hamming', fs=fs)
b_iir, a_iir = signal.butter(4, 200, fs=fs)

y_fir = signal.lfilter(b_fir, [1], x)
y_iir = signal.lfilter(b_iir, a_iir, x)

plt.figure(figsize=(12, 5))
plt.plot(t, x, 'gray', alpha=0.4, label='Исходный')
plt.plot(t, y_fir, label='КИХ')
plt.plot(t, y_iir, label='БИХ')
plt.xlim(0.015, 0.045)
plt.title('Фильтрация импульса с помехой')
plt.grid(True)
plt.legend()
plt.show()



**Вопросы:** Какой фильтр лучше сохранил форму импульсов? Какой лучше подавил помеху?

1) Форму импульса лучше сохраняет КИХ-фильтр, потому что у него линейная фаза
2) Помеху 400 Гц чуть сильнее подавляет БИХ-фильтр, но он сильнее меняет форму фронтов
